In [1]:
import os
import shutil
import random
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIGURATION — adjust splits as needed
# ─────────────────────────────────────────────
DATASET_ROOT   = r"D:\Mold Studies\Mould Studies Multiclass"
OUTPUT_ROOT    = DATASET_ROOT  # splits go into the same root folder

TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
TEST_RATIO     = 0.15

RANDOM_SEED    = 27
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

# ─────────────────────────────────────────────
# STEP 1 — Discover classes
# ─────────────────────────────────────────────
dataset_path = Path(DATASET_ROOT)

classes = sorted([
    d.name for d in dataset_path.iterdir()
    if d.is_dir()
])

print(f"✅ Found {len(classes)} classes:\n")
for i, cls in enumerate(classes, 1):
    print(f"  {i:>2}. {cls}")

# ─────────────────────────────────────────────
# STEP 2 — Validate ratios
# ─────────────────────────────────────────────
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, \
    "Train + Val + Test ratios must sum to 1.0"

# ─────────────────────────────────────────────
# STEP 3 — Split and copy files
# ─────────────────────────────────────────────
random.seed(RANDOM_SEED)

split_summary = []

for cls in classes:
    src_folder = dataset_path / cls

    # Collect all images
    images = [
        f for f in src_folder.iterdir()
        if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
    ]

    if not images:
        print(f"⚠️  No images found in class: {cls}")
        continue

    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * TRAIN_RATIO)
    n_val   = int(n_total * VAL_RATIO)
    n_test  = n_total - n_train - n_val  # remainder goes to test

    splits = {
        "train": images[:n_train],
        "val":   images[n_train : n_train + n_val],
        "test":  images[n_train + n_val:]
    }

    for split_name, split_files in splits.items():
        dest_folder = Path(OUTPUT_ROOT) / split_name / cls
        dest_folder.mkdir(parents=True, exist_ok=True)

        for img_path in split_files:
            shutil.copy2(img_path, dest_folder / img_path.name)

    split_summary.append({
        "class":  cls,
        "total":  n_total,
        "train":  n_train,
        "val":    n_val,
        "test":   n_test
    })

# ─────────────────────────────────────────────
# STEP 4 — Summary table
# ─────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"{'Class':<25} {'Total':>7} {'Train':>7} {'Val':>6} {'Test':>6}")
print(f"{'─'*55}")

for row in split_summary:
    print(f"{row['class']:<25} {row['total']:>7} {row['train']:>7} {row['val']:>6} {row['test']:>6}")

totals = {k: sum(r[k] for r in split_summary) for k in ["total", "train", "val", "test"]}
print(f"{'─'*55}")
print(f"{'TOTAL':<25} {totals['total']:>7} {totals['train']:>7} {totals['val']:>6} {totals['test']:>6}")
print(f"\n✅ Split complete! Output saved to:\n   {OUTPUT_ROOT}")

# ─────────────────────────────────────────────
# STEP 5 — Delete original class folders
# ─────────────────────────────────────────────
print("\n🗑️  Removing original class folders...")

PROTECTED = {"train", "val", "test"}

for cls in classes:
    if cls.lower() in PROTECTED:
        print(f"   Skipped (protected): {cls}")
        continue
    original_folder = dataset_path / cls
    if original_folder.exists():
        shutil.rmtree(original_folder)
        print(f"   Deleted: {original_folder}")

print("\n✅ Done. Folder now contains only: train / val / test")

✅ Found 11 classes:

   1. Blackberry
   2. Blueberry
   3. Carrots
   4. Cheese
   5. Cream Cheese
   6. Mixed Bread
   7. Onion
   8. Orange
   9. Raspberry
  10. Toast
  11. Tomatoes

───────────────────────────────────────────────────────
Class                       Total   Train    Val   Test
───────────────────────────────────────────────────────
Blackberry                    168     117     25     26
Blueberry                     168     117     25     26
Carrots                       337     235     50     52
Cheese                        203     142     30     31
Cream Cheese                  334     233     50     51
Mixed Bread                   240     168     36     36
Onion                         685     479    102    104
Orange                        471     329     70     72
Raspberry                     168     117     25     26
Toast                         683     478    102    103
Tomatoes                      864     604    129    131
─────────────────────────────